In [ ]:
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!pip install -r requirements.txt
!pip install kornia==0.7.1 onnxruntime-gpu insightface
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git custom_nodes/ComfyUI-Manager
!git clone https://github.com/Lightricks/ComfyUI-LTXVideo.git custom_nodes/ComfyUI-LTXVideo
!pip install -r custom_nodes/ComfyUI-LTXVideo/requirements.txt
!git clone https://github.com/Gourieff/ComfyUI-ReActor.git custom_nodes/ComfyUI-ReActor
!pip install -r custom_nodes/ComfyUI-ReActor/requirements.txt
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
import os
print("Downloading LTX-Video Models...")
!mkdir -p models/checkpoints/LTX-Video
!wget -q -O models/checkpoints/LTX-Video/ltx-video-2b-v0.9.1.safetensors https://huggingface.co/Lightricks/LTX-Video/resolve/main/ltx-video-2b-v0.9.1.safetensors
print("Downloading T5 FP8 Text Encoder...")
!mkdir -p models/clip
!wget -q -O models/clip/t5xxl_fp8_e4m3fn.safetensors https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors
print("Copying InsightFace models from Kaggle Dataset...")
!mkdir -p models/insightface
!find /kaggle/input -name "inswapper_128.onnx" -exec cp {} models/insightface/inswapper_128.onnx \;
print("Download Complete!")

In [ ]:
import subprocess
import threading
import time
import socket

def tunnel_thread():
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', 8188))
        if result == 0:
            break
        sock.close()
    print("\n🚀 Launching Cloudflare Tunnel...\n")
    p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l:
            print("\n\n✨ ComfyUI is LIVE! Click this link to open: ", l[l.find("http"):], "\n\n")

threading.Thread(target=tunnel_thread, daemon=True).start()

# Start ComfyUI on 0.0.0.0 and allow all CORS to prevent localhost proxy blocks
!python main.py --fp8_e4m3fn-unet --fp8_e4m3fn-text-enc --listen 0.0.0.0 --enable-cors-header "*"